# NTSAMAISE: YOLO11 road-sign detection
In Colab, select **Runtime > Change runtime type > GPU**, then run cells in order.

Prepare a bounding-box detection dataset in Google Drive with `images/train`, `images/val`, `images/test` and corresponding `labels/train`, `labels/val`, `labels/test` folders. Each image needs a matching YOLO `.txt` label: `class_id x_center y_center width height`, with coordinates normalized to 0–1. Class IDs must match your `data.yaml` names. A classification-only dataset of cropped signs needs detection annotations before use here.

Use separate train, validation and test sets. Keep images of the same physical sign, journey or video together in one split to prevent leakage. Include local signs, varied lighting/distances and images without target signs. Do not mirror directional signs.

Your YAML should use an absolute Colab dataset path, for example:
```yaml
path: /content/drive/MyDrive/road_signs
train: images/train
val: images/val
test: images/test
names:
  0: stop
  1: yield
```
The names above are examples: preserve the actual annotation class IDs and names.

References: [YOLO11](https://docs.ultralytics.com/models/yolo11/), [dataset format](https://docs.ultralytics.com/datasets/detect/), [training](https://docs.ultralytics.com/modes/train/).

In [ ]:
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import torch
import ultralytics
from ultralytics import YOLO

DATA = Path('/content/drive/MyDrive/road_signs/data.yaml') # Change to your YAML
OUTPUT = Path('/content/drive/MyDrive/ntsamaise_road_sign_runs')
assert DATA.is_file(), f'Dataset YAML missing: {DATA}'
assert torch.cuda.is_available(), 'Enable a GPU runtime before training.'
print('GPU:', torch.cuda.get_device_name(0))
print('Ultralytics:', ultralytics.__version__, 'PyTorch:', torch.__version__)

In [ ]:
from datetime import datetime, timezone
import subprocess

OUTPUT.mkdir(parents=True, exist_ok=True)
RUN = 'yolo11n_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
(OUTPUT / f'{RUN}_requirements.txt').write_text(
    subprocess.check_output(['pip', 'freeze'], text=True)
)
model = YOLO('yolo11n.pt')
model.train(
    data=str(DATA), epochs=100, imgsz=640, batch=8, device=0,
    workers=2, seed=42, patience=20,
    fliplr=0.0, flipud=0.0,
    project=str(OUTPUT), name=RUN, exist_ok=False,
)
best = Path(model.trainer.best)
assert best.is_file(), f'Missing trained weights: {best}'
print('Best weights:', best)

If GPU memory runs out, reduce `batch` to 4 or 2. Epochs and image size are starting settings, not accuracy guarantees. Drive retains completed checkpoints if Colab disconnects. To resume an interrupted run in a fresh session after installation and mounting Drive, use `YOLO('/content/drive/MyDrive/ntsamaise_road_sign_runs/YOUR_RUN/weights/last.pt').train(resume=True)`.

Review validation results and per-class errors while developing. Run the following held-out test evaluation only when model choices are finalized; do not tune against the test set.

In [ ]:
import yaml
config = yaml.safe_load(DATA.read_text())
assert config.get('test'), 'Provide a separate held-out test split in data.yaml.'
trained = YOLO(str(best))
metrics = trained.val(data=str(DATA), split='test', imgsz=640, device=0,
                      project=str(OUTPUT), name=RUN + '_test')
print('Test mAP50-95:', metrics.box.map)
print('Test mAP50:', metrics.box.map50)
print('Per-class mAP50-95:', metrics.box.maps)

The trained `weights/best.pt`, plots, metrics and run settings are saved in Google Drive. Inspect missed signs and false detections on representative Lesotho scenes before considering Android integration.

NTSAMAISE currently uses MediaPipe EfficientDet-Lite0. These YOLO weights are not a drop-in replacement: export, compatible preprocessing, output decoding and phone performance/accuracy testing are separate work. This notebook does not modify the Android app. No training or accuracy result exists until you run it with your dataset.